# KrishiVision Training Pipeline

This notebook runs the complete training pipeline for KrishiVision on Google Colab with T4 GPU.

## Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow==2.15.0 tf-keras-vis==0.8.0 opencv-python==4.9.0.80 \
    albumentations==1.3.1 scikit-learn==1.3.2 matplotlib==3.8.2 \
    seaborn==0.13.1 kaggle==1.5.16 tqdm==4.66.1 pyyaml==6.0.1

# Check GPU
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TF Version:", tf.__version__)

## Configure Kaggle API

In [ ]:
# Upload kaggle.json or set credentials
import os
import json

# Option 1: Upload kaggle.json via Colab file browser
# Option 2: Set environment variables
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_key'

# Verify
!mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

print("Kaggle configured:", os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')))

## Download and Prepare Data

In [ ]:
# Download PlantVillage dataset
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data --unzip

# Download PlantDoc for evaluation
!wget -q https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset/archive/refs/heads/main.zip -O /content/data/plantdoc.zip
!unzip -q /content/data/plantdoc.zip -d /content/data/plantdoc

print("Data downloaded")

In [ ]:
# Organize PlantVillage into train/val/test splits
import shutil
from sklearn.model_selection import train_test_split
from pathlib import Path

CLASS_NAMES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry___Powdery_mildew", "Cherry___healthy",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot", "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight",
    "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
]

source_dir = Path("/content/data/PlantVillage")
if not source_dir.exists():
    source_dir = Path("/content/data/plantvillage-dataset/PlantVillage")

train_dir = Path("/content/data/train")
val_dir = Path("/content/data/val")
test_dir = Path("/content/data/test")

for d in [train_dir, val_dir, test_dir]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

all_images = []
all_labels = []

for class_name in CLASS_NAMES:
    class_path = source_dir / class_name
    if not class_path.exists():
        continue
    images = list(class_path.glob("*.jpg")) + list(class_path.glob("*.JPG")) + list(class_path.glob("*.png"))
    for img_path in images:
        all_images.append(str(img_path))
        all_labels.append(class_name)

print(f"Total images: {len(all_images)}")

train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, train_size=0.7, stratify=all_labels, random_state=42
)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

for imgs, labels, target_dir in [
    (train_imgs, train_labels, train_dir),
    (val_imgs, val_labels, val_dir),
    (test_imgs, test_labels, test_dir)
]:
    for img_path, label in zip(imgs, labels):
        label_dir = target_dir / label
        label_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img_path, label_dir / Path(img_path).name)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

## Phase 1: Baseline Training (Frozen Base)

In [ ]:
%cd /content
!git clone https://github.com/knokvik/krishivision.git 2>/dev/null || true
%cd /content/krishivision

# Quick test with --quick flag (2 epochs)
!python -m backend.ml.train_baseline --quick

# For full training, run without --quick:
# !python -m backend.ml.train_baseline

## Phase 2: Fine-tuning (Unfreeze Last 30 Layers)

In [ ]:
# Fine-tune from Phase 1 checkpoint
!python -m backend.ml.train_finetune --checkpoint /content/krishivision/backend/ml/model.phase1.keras --quick

# For full training:
# !python -m backend.ml.train_finetune --checkpoint /content/krishivision/backend/ml/model.phase1.keras

## Evaluation

In [ ]:
# Evaluate on PlantVillage test set and PlantDoc
!python -m backend.ml.evaluate --model /content/krishivision/backend/ml/model.keras --plantdoc --ablation

## Export to TFLite (int8 Quantized)

In [ ]:
# Export quantized model
!python -m backend.ml.export_tflite --quantization int8

# Check model size
import os
size_mb = os.path.getsize('/content/krishivision/backend/ml/model.tflite') / (1024*1024)
print(f"TFLite model size: {size_mb:.2f} MB")

## Test Grad-CAM

In [ ]:
# Test Grad-CAM on a sample image
!python -m backend.ml.gradcam --model /content/krishivision/backend/ml/model.keras --image /content/data/test/Tomato___Early_blight/*.jpg --num 1

## Package for Deployment

In [ ]:
# Copy model files to deployment location
!mkdir -p /content/deploy/backend/ml
!cp /content/krishivision/backend/ml/model.tflite /content/deploy/backend/ml/
!cp /content/krishivision/backend/ml/model.keras /content/deploy/backend/ml/
!cp -r /content/krishivision/backend/remedies /content/deploy/backend/
!cp /content/krishivision/backend/config.py /content/deploy/backend/
!cp /content/krishivision/backend/reject.py /content/deploy/backend/
!cp /content/krishivision/backend/gradcam.py /content/deploy/backend/
!cp /content/krishivision/backend/pdf_report.py /content/deploy/backend/
!cp /content/krishivision/backend/main.py /content/deploy/backend/

print("Deployment package ready at /content/deploy")